In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
df = pd.read_csv('fashion-mnist_train.csv')
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
19199,0,0,0,0,0,0,1,0,0,84,...,80,53,0,0,1,0,0,0,0,0
17574,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
25941,6,0,0,0,0,0,0,0,0,0,...,30,0,0,0,0,0,0,0,0,0
29237,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
56851,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,78,63,1,0,0,0


In [5]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [9]:
from numpy import dtype
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1, 1, 28, 28)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):

        return len(self.features)

    def __getitem__(self, index):

        return self.features[index], self.labels[index]

In [10]:
train_dataset = CustomDataset(X_train, y_train)

In [11]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [13]:
class MyNN(nn.Module):

    def __init__(self, input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            
            nn.Linear(64, 10)
            
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [14]:
epochs = 100
learning_rate = 0.01

In [15]:
model = MyNN(1)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [16]:
for epoch in range(epochs):

    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epoch_loss += loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)

    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")




Epoch: 1, Loss: 0.645301487594843
Epoch: 2, Loss: 0.3857074027011792
Epoch: 3, Loss: 0.32687767408788204
Epoch: 4, Loss: 0.28987051425129173
Epoch: 5, Loss: 0.2643057978575428
Epoch: 6, Loss: 0.24223418862496812
Epoch: 7, Loss: 0.22998863366805017
Epoch: 8, Loss: 0.21234340214232603
Epoch: 9, Loss: 0.20052464481132726
Epoch: 10, Loss: 0.19125346327635148
Epoch: 11, Loss: 0.17765039778935413
Epoch: 12, Loss: 0.17043269584390025
Epoch: 13, Loss: 0.1565553993411983
Epoch: 14, Loss: 0.1487970598693937
Epoch: 15, Loss: 0.14004866172621647
Epoch: 16, Loss: 0.13164937286901598
Epoch: 17, Loss: 0.1267113775294274
Epoch: 18, Loss: 0.12190257936203852
Epoch: 19, Loss: 0.11429235982894898
Epoch: 20, Loss: 0.10857719529637445
Epoch: 21, Loss: 0.10958207163168117
Epoch: 22, Loss: 0.1041883237324655
Epoch: 23, Loss: 0.09688088847448428
Epoch: 24, Loss: 0.08994054888328537
Epoch: 25, Loss: 0.08638612115181361
Epoch: 26, Loss: 0.08459261841668438
Epoch: 27, Loss: 0.08264562961641544
Epoch: 28, Loss: 0

In [17]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [18]:
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    
print(correct/total)

0.922


In [19]:
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()
    
print(correct/total)

0.9971875
